# SteelVision AI — Exploration Notebook

This notebook demonstrates the core SteelVision AI pipeline:
- Dataset validation & statistics
- Running inference on a sample image
- Viewing analytics
- Querying the Digital Steel Passport database

> **Requires:** `pip install -r requirements.txt` and a trained model at `models/best.pt`.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

## 1. System & Hardware Info

In [ ]:
from src.utils.hardware import get_hardware_info, print_hardware_info

print_hardware_info()

## 2. Dataset Validation

In [ ]:
from src.data.validate_dataset import DatasetValidator

# Adjust the path to your NEU-DET dataset
dataset_root = ROOT / "data" / "neu_det"
config_path  = ROOT / "configs" / "dataset.yaml"

if dataset_root.exists():
    validator = DatasetValidator(dataset_root, config_path)
    report = validator.validate()
    print(f"Total images  : {report.get('total_images', 0)}")
    print(f"Total labels  : {report.get('total_labels', 0)}")
    print(f"Missing labels: {report.get('missing_labels', 0)}")
    print(f"Errors        : {report.get('errors_found', 0)}")
else:
    print(f"[INFO] Dataset not found at {dataset_root}.")
    print("       Run: python scripts/prepare_dataset.py --source data/raw/NEU-DET --output data/neu_det")

## 3. Analytics Demo (synthetic detections)

In [ ]:
from src.analytics.defect_analytics import DefectAnalytics

analytics = DefectAnalytics()

synthetic_dets = [
    {"class_name": "crazing",       "confidence": 0.82, "region": "Top-left",    "bbox_area_px2": 1200.0, "frame_percentage": 0.5,  "cx_norm": 0.1, "cy_norm": 0.1},
    {"class_name": "scratches",     "confidence": 0.55, "region": "Center",      "bbox_area_px2":  800.0, "frame_percentage": 0.3,  "cx_norm": 0.5, "cy_norm": 0.5},
    {"class_name": "crazing",       "confidence": 0.30, "region": "Top-left",    "bbox_area_px2":  400.0, "frame_percentage": 0.2,  "cx_norm": 0.2, "cy_norm": 0.2},
    {"class_name": "inclusion",     "confidence": 0.70, "region": "Bottom-right","bbox_area_px2":  600.0, "frame_percentage": 0.25, "cx_norm": 0.9, "cy_norm": 0.9},
    {"class_name": "pitted_surface","confidence": 0.88, "region": "Center",      "bbox_area_px2":  900.0, "frame_percentage": 0.4,  "cx_norm": 0.5, "cy_norm": 0.6},
]

analytics.add_detections(synthetic_dets)
summary = analytics.get_summary()

print(f"Total detections      : {summary['total_detections']}")
print(f"Most frequent class   : {summary['most_frequent_class']}")
print(f"Most affected region  : {summary['most_affected_region']}")
print(f"Avg confidence overall: {summary['avg_confidence_overall']:.3f}")
print(f"Low confidence count  : {summary['low_confidence_count']}")
print()
print("Class distribution:")
for cls, cnt in summary['class_distribution'].items():
    if cnt > 0:
        print(f"  {cls:<20}: {cnt}")

## 4. Digital Steel Passport Demo

In [ ]:
from src.passport.passport_manager import PassportManager
import tempfile, json

# Use a temp DB for demo
with tempfile.TemporaryDirectory() as tmp:
    pm = PassportManager(db_path=Path(tmp) / "demo.db")

    insp_id = pm.start_inspection(
        input_source="notebook_demo",
        model_name="yolov8n.pt",
        backend="PyTorch",
        conf_threshold=0.30,
        iou_threshold=0.45,
        operator_notes="Notebook demonstration run.",
    )
    print(f"Started inspection: {insp_id}")

    summary = {
        "total_frame_detections": 5,
        "unique_tracked_defects": 3,
        "avg_confidence": 0.65,
        "low_conf_count": 1,
        "class_counts": {"crazing": 2, "scratches": 1, "pitted_surface": 1, "inclusion": 1},
        "region_counts": {"Top-left": 2, "Center": 3},
        "pattern_count": 0,
        "longest_recurring_sequence": 0,
        "most_frequent_class": "crazing",
        "most_affected_region": "Center",
        "most_recurring_class": None,
        "highest_recurrence_region": None,
    }

    passport = pm.finalize_inspection(insp_id, summary)
    print(f"Inspection status: {passport.get('inspection_status')}")
    print(f"Status reason:     {passport.get('status_reason')}")
    print(f"Duration (s):      {passport.get('duration_seconds', 0):.2f}")
    print()

    # List all inspections
    records = pm.list_inspections()
    print(f"Total inspections in DB: {len(records)}")

## 5. Live Inference (requires trained model)

Uncomment the cell below once you have a trained model at `models/best.pt`.

In [ ]:
# from src.inference.predictor import UltralyticsPredictor
# import cv2
# from IPython.display import Image as IPImage
#
# weights = ROOT / "models" / "best.pt"
# img_path = ROOT / "data" / "neu_det" / "images" / "val" / "example.jpg"   # adjust path
#
# if weights.exists() and img_path.exists():
#     predictor = UltralyticsPredictor(weights_path=weights, conf=0.30, iou=0.45)
#     result = predictor.predict_image(img_path)
#     print(f"Detections : {result['num_detections']}")
#     print(f"Latency ms : {result['latency_ms']}")
#     print(f"Status     : {result['inspection']}")
#
#     save_path = ROOT / "results" / "predictions" / "notebook_demo.jpg"
#     save_path.parent.mkdir(parents=True, exist_ok=True)
#     cv2.imwrite(str(save_path), result['annotated_image'])
#     IPImage(filename=str(save_path))
# else:
#     print("[INFO] No model or image found. Train first: python scripts/train.py")
print("Inference cell is commented out. Uncomment after training.")

---
## Summary

| Component | Status |
|-----------|--------|
| Hardware info | ✅ |
| Dataset validation | ✅ (requires dataset) |
| Defect analytics | ✅ |
| Digital Passport | ✅ |
| Live inference | Requires `models/best.pt` |

Run `streamlit run demo/app.py` for the full interactive dashboard.